# Creating Agents in AutoGen

This notebook is for GenAI classroom training.  
It explains how to create agents, assign roles, configure system messages, and simulate agent collaboration.

> Many cells are executable without an API key.  
> Live AutoGen examples are included separately and will run only when the required API key is available.

## 1. What is an Agent?

An agent is a role-based AI worker.

In AutoGen, an agent usually has:

- A name
- A role or purpose
- A system message
- A model client
- Optional tools
- Optional rules for collaboration

Example roles:

- Requirement Analyzer
- Test Case Generator
- Code Generator
- Reviewer
- Debugging Assistant

## 2. Simple Agent Class

Before using the real AutoGen library, let us create a basic Python agent to understand the concept.

In [ ]:
class BasicAgent:
    def __init__(self, name, role, system_message):
        self.name = name
        self.role = role
        self.system_message = system_message

    def describe(self):
        return {
            "name": self.name,
            "role": self.role,
            "system_message": self.system_message
        }

agent = BasicAgent(
    name="TestCaseAgent",
    role="Creates manual and automation test cases",
    system_message="You are an expert QA engineer. Create clear and complete test cases."
)

agent.describe()

## 3. Why System Message is Important

The system message controls the behavior of the agent.

Example:

Instead of saying only:

> You are helpful.

We can say:

> You are a senior SDET. Generate test cases with positive, negative, boundary, and edge scenarios.

A good system message gives better output.

In [ ]:
system_messages = {
    "weak": "You are helpful.",
    "strong": "You are a senior SDET. Generate test cases with positive, negative, boundary, and edge scenarios."
}

for key, value in system_messages.items():
    print(f"{key.upper()} SYSTEM MESSAGE:")
    print(value)
    print()

## 4. Creating Different Agents for Different Roles

In [ ]:
requirement_agent = BasicAgent(
    name="RequirementAgent",
    role="Understands user stories and acceptance criteria",
    system_message="Analyze requirements and identify business rules, validations, and assumptions."
)

testcase_agent = BasicAgent(
    name="TestCaseAgent",
    role="Creates test cases",
    system_message="Create test cases covering positive, negative, boundary, and edge cases."
)

review_agent = BasicAgent(
    name="ReviewAgent",
    role="Reviews test coverage",
    system_message="Review generated test cases and identify missing scenarios."
)

agents = [requirement_agent, testcase_agent, review_agent]

for agent in agents:
    print(agent.describe())
    print("-" * 80)

## 5. Agent Naming Best Practices

Use meaningful names.

Good names:

- `requirement_analyzer`
- `test_case_generator`
- `automation_code_writer`
- `test_reviewer`

Avoid vague names:

- `agent1`
- `bot`
- `helper`

In [ ]:
good_agent_names = [
    "requirement_analyzer",
    "test_case_generator",
    "automation_code_writer",
    "test_reviewer"
]

bad_agent_names = [
    "agent1",
    "bot",
    "helper"
]

print("Good agent names:", good_agent_names)
print("Bad agent names:", bad_agent_names)

## 6. Simulating Agent Response

This example simulates how agents can respond based on their role.

In [ ]:
class RuleBasedAgent:
    def __init__(self, name, role):
        self.name = name
        self.role = role

    def respond(self, task):
        if "requirement" in self.role.lower():
            return f"{self.name}: I will identify rules and assumptions from: {task}"
        elif "test" in self.role.lower():
            return f"{self.name}: I will create test cases for: {task}"
        elif "review" in self.role.lower():
            return f"{self.name}: I will review coverage and missing scenarios for: {task}"
        else:
            return f"{self.name}: I will help with: {task}"

task = "Login should work with valid credentials and show error for invalid credentials."

agents = [
    RuleBasedAgent("RequirementAgent", "Requirement analysis"),
    RuleBasedAgent("TestCaseAgent", "Test case creation"),
    RuleBasedAgent("ReviewAgent", "Review coverage")
]

for agent in agents:
    print(agent.respond(task))

## 7. Creating an Agent Configuration Dictionary

In real projects, keeping agent configuration in a dictionary or JSON file makes the system easier to maintain.

In [ ]:
agent_configs = {
    "RequirementAgent": {
        "role": "Requirement analysis",
        "system_message": "Extract rules, validations, and assumptions from requirements."
    },
    "TestCaseAgent": {
        "role": "Test case generation",
        "system_message": "Generate complete functional test cases."
    },
    "AutomationAgent": {
        "role": "Automation script generation",
        "system_message": "Generate WebDriverIO or Selenium automation skeletons."
    }
}

for agent_name, config in agent_configs.items():
    print(agent_name)
    print("Role:", config["role"])
    print("System Message:", config["system_message"])
    print()

## 8. Creating Agents Dynamically from Configuration

In [ ]:
dynamic_agents = []

for agent_name, config in agent_configs.items():
    dynamic_agents.append(
        BasicAgent(
            name=agent_name,
            role=config["role"],
            system_message=config["system_message"]
        )
    )

for agent in dynamic_agents:
    print(agent.describe())

## 9. Adding Tools to an Agent

A tool is a Python function that an agent can use.

Example tools:

- Calculator
- CSV reader
- API caller
- Database query executor
- Test data generator

In [ ]:
def generate_test_data(username_prefix: str, count: int):
    return [f"{username_prefix}_{i}" for i in range(1, count + 1)]

test_users = generate_test_data("testuser", 5)
print(test_users)

## 10. Tool-Enabled Agent Example

This is a mock example that does not need an API key.

In [ ]:
class ToolEnabledAgent:
    def __init__(self, name, tools=None):
        self.name = name
        self.tools = tools or {}

    def use_tool(self, tool_name, *args):
        if tool_name not in self.tools:
            return f"Tool '{tool_name}' is not available."
        return self.tools[tool_name](*args)

agent = ToolEnabledAgent(
    name="TestDataAgent",
    tools={"generate_test_data": generate_test_data}
)

agent.use_tool("generate_test_data", "login_user", 3)

## 11. Install AutoGen Packages

Run this only once if AutoGen is not installed.

In [ ]:
# Uncomment and run if needed:
# !pip install -U autogen-agentchat autogen-ext[openai]

## 12. Check AutoGen Installation

In [ ]:
try:
    import autogen_agentchat
    print("AutoGen AgentChat is installed.")
except ImportError:
    print("AutoGen AgentChat is not installed. Run the pip install cell above.")

## 13. Creating a Real AutoGen AssistantAgent

This example requires an OpenAI API key.

Set the key in PowerShell:

```powershell
$env:OPENAI_API_KEY="your_api_key_here"
```

In [ ]:
import os

async def create_simple_autogen_agent():
    try:
        from autogen_agentchat.agents import AssistantAgent
        from autogen_ext.models.openai import OpenAIChatCompletionClient
    except ImportError:
        print("Required AutoGen packages are not installed.")
        return

    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found. Skipping live AutoGen call.")
        return

    model_client = OpenAIChatCompletionClient(
        model="gpt-4o-mini",
        api_key=os.getenv("OPENAI_API_KEY")
    )

    assistant = AssistantAgent(
        name="testcase_agent",
        model_client=model_client,
        system_message="You are a senior QA engineer. Generate concise and clear test cases."
    )

    result = await assistant.run(
        task="Create 5 test cases for login functionality."
    )

    print(result)

# In Jupyter, run:
# await create_simple_autogen_agent()

## 14. Creating Multiple Real AutoGen Agents

This shows the structure for multiple AutoGen agents.

The exact collaboration/team setup can vary based on AutoGen version.

In [ ]:
async def create_multiple_autogen_agents():
    try:
        from autogen_agentchat.agents import AssistantAgent
        from autogen_ext.models.openai import OpenAIChatCompletionClient
    except ImportError:
        print("Required AutoGen packages are not installed.")
        return

    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found. Skipping live AutoGen call.")
        return

    model_client = OpenAIChatCompletionClient(
        model="gpt-4o-mini",
        api_key=os.getenv("OPENAI_API_KEY")
    )

    requirement_agent = AssistantAgent(
        name="requirement_agent",
        model_client=model_client,
        system_message="Analyze requirements and identify rules and assumptions."
    )

    testcase_agent = AssistantAgent(
        name="testcase_agent",
        model_client=model_client,
        system_message="Generate functional test cases."
    )

    review_agent = AssistantAgent(
        name="review_agent",
        model_client=model_client,
        system_message="Review test cases and identify missing scenarios."
    )

    print("Created agents:")
    print(requirement_agent.name)
    print(testcase_agent.name)
    print(review_agent.name)

# In Jupyter, run:
# await create_multiple_autogen_agents()

## 15. Creating an Agent with Anthropic Claude

Install package if needed:

```python
!pip install -U autogen-agentchat autogen-ext[anthropic]
```

Set environment variable:

```powershell
$env:ANTHROPIC_API_KEY="your_api_key_here"
```

In [ ]:
async def create_anthropic_autogen_agent():
    try:
        from autogen_agentchat.agents import AssistantAgent
        from autogen_ext.models.anthropic import AnthropicChatCompletionClient
    except ImportError:
        print("Anthropic AutoGen extension is not installed.")
        return

    if not os.getenv("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY not found. Skipping live Anthropic call.")
        return

    model_client = AnthropicChatCompletionClient(
        model="claude-3-5-sonnet-latest",
        api_key=os.getenv("ANTHROPIC_API_KEY")
    )

    assistant = AssistantAgent(
        name="claude_testcase_agent",
        model_client=model_client,
        system_message="You are a QA automation trainer. Explain agent creation clearly."
    )

    result = await assistant.run(
        task="Explain how to create agents in AutoGen with a simple example."
    )

    print(result)

# In Jupyter, run:
# await create_anthropic_autogen_agent()

## 16. Classroom Exercise

Create one more agent called `DefectAnalysisAgent`.

It should:

- Read failed test details
- Identify possible root cause
- Suggest debugging steps

In [ ]:
# Exercise solution sample

defect_analysis_agent = BasicAgent(
    name="DefectAnalysisAgent",
    role="Analyzes failed tests and suggests root cause",
    system_message="You are an expert test automation debugger. Analyze failures and suggest debugging steps."
)

defect_analysis_agent.describe()

## 17. Mini Project: Agent Design for QA Automation

Design agents for this workflow:

Requirement → Test Cases → Automation Script → Review → Final Output

In [ ]:
qa_agent_design = [
    {
        "agent": "RequirementAgent",
        "responsibility": "Understand requirement and acceptance criteria"
    },
    {
        "agent": "TestCaseAgent",
        "responsibility": "Generate positive, negative, and edge test cases"
    },
    {
        "agent": "AutomationAgent",
        "responsibility": "Generate automation script skeleton"
    },
    {
        "agent": "ReviewAgent",
        "responsibility": "Review coverage and improve quality"
    }
]

for item in qa_agent_design:
    print(f"{item['agent']}: {item['responsibility']}")

## 18. Key Takeaways

- Agents should have clear names and responsibilities.
- System messages are very important.
- Different agents should be created for different tasks.
- Tools can be attached to agents for extra capability.
- For teaching, start with mock agents before using real LLM agents.
- AutoGen agents can be connected to OpenAI, Anthropic, and other model providers.